# Echo: voice-cloning runner

Speaks `TEXT` in the voice of a short reference recording, given its transcript. The recording is encoded with the Higgs codec on the fly; [fixed_reference.ipynb](fixed_reference.ipynb) is the same pipeline for a reference encoded in advance.

It runs on its own: nothing is imported from the Echo codebase. Requirements: `torch`, `torchaudio`, `transformers` (5.3 or later), `safetensors`, `soundfile`, `numpy`.

**The model.** Echo is OmniVoice pruned to 297M parameters and retrained: a masked-diffusion TTS model. Audio is Higgs codec tokens, 25 frames per second, each frame 8 codebooks from coarse (0) to fine (7). One bidirectional transformer reads one sequence:

```
[ prompt + reference text + text | reference frames ][ target frames ]
                 prefix                                    target
```

The target starts fully masked and is filled in over several passes. Two properties of this checkpoint keep that cheap:
- **Prefix blocking:** the prefix never attends to the target, so it runs once and its keys and values are cached. Each pass then runs over the target frames only.
- **Distilled guidance:** classifier-free guidance is distilled into the weights, so each pass is a single forward, with no unconditional branch.

**The sampler: sNcb2 + 2**, with N = `STEPS` ([two_stage_sampling.md](../../knowledge/inference/two_stage_sampling.md), [polish_pass.md](../../knowledge/inference/polish_pass.md)):
1. **N main passes decide codebooks 0-1**, which carry content and timing. Each pass commits the masked slots that score highest (confidence minus a per-codebook penalty, plus Gumbel noise), on a schedule that saves most commits for the last pass. Every pass but the last then re-masks the least confident earlier commits, half as many as it just made.
2. **2 polish passes decide codebooks 2-7**, with codebooks 0-1 finished and extra attention on the reference frames: codebooks 2-3 first, then 4-7, guided against the first polish pass's prediction.

One generation costs 1 prefix pass and N + 2 target passes.

In [ ]:
from pathlib import Path

ROOT = Path("../..")  # the repo root, seen from this notebook
CHECKPOINT = ROOT / "runs/final/fp32"  # config.json, model.safetensors and the text tokenizer
CODEC = "eustlb/higgs-audio-v2-tokenizer"  # the Higgs codec: a Hugging Face hub id or a local copy
REFERENCE = ROOT / "data/test/german/audio/de_1_anna.wav"  # a clean recording of 3-10 s
REFERENCE_TEXT = "Die Temperatur eines Kindes kann Ihnen einen wertvollen Einblick in dessen Gesundheit verschaffen, aber diese korrekt zu messen, kann leicht zu einer Herausforderung werden."
TEXT = "Das Ohrthermometer liefert in nur einer Sekunde ein präzises Ergebnis, auch wenn Ihr Kind schläft."
LANGUAGE = "de"
STEPS = 8  # N main passes
SEED = 0
OUTPUT = "voice_cloning.wav"

## Load

`Echo` holds the checkpoint's weights: a Qwen3 backbone, the text embedding, and the audio embeddings and heads. The text tokenizer comes with the checkpoint. Both halves of the codec are used: the encoder for the reference, the decoder for the output.

In [ ]:
import json
import math
import unicodedata
from typing import Union

import numpy as np
import soundfile as sf
import torch
import torchaudio
from IPython.display import Audio
from safetensors.torch import load_file
from transformers import DynamicCache, HiggsAudioV2TokenizerModel, Qwen2Tokenizer, Qwen3Config, Qwen3Model

torch.set_grad_enabled(False)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class Echo(torch.nn.Module):
    """The checkpoint's weights: a Qwen3 backbone, its text embedding, and the audio embeddings and heads."""

    def __init__(self, checkpoint: Union[str, Path]) -> None:
        super().__init__()
        config = json.loads(Path(checkpoint, "config.json").read_text())
        weights = load_file(Path(checkpoint, "model.safetensors"))
        self.llm = Qwen3Model(Qwen3Config.from_dict(config["llm_config"]))
        hidden = self.llm.config.hidden_size
        self.audio_embeddings = torch.nn.Embedding(8 * 1025, hidden)  # 8 codebooks x (1024 codes + the mask)
        self.audio_heads = torch.nn.Linear(hidden, 8 * 1025, bias=False)

        # A pruned checkpoint stores the text embedding factorized: a [vocab, rank] table and a rank -> hidden projection.
        if "llm.embed_tokens.table.weight" in weights:
            table, project = weights.pop("llm.embed_tokens.table.weight"), weights.pop("llm.embed_tokens.project.weight")
            weights["llm.embed_tokens.weight"] = table @ project.T
        # Centering each codebook's head rows leaves its softmax unchanged, and keeps the logits small enough for bf16.
        heads = weights["audio_heads.weight"].view(8, 1025, hidden)
        weights["audio_heads.weight"] = (heads - heads.mean(1, keepdim=True)).view(8 * 1025, hidden)
        weights.pop("codebook_layer_offsets", None)
        self.load_state_dict(weights)


model = Echo(CHECKPOINT).to(DEVICE).eval()
tokenizer = Qwen2Tokenizer.from_pretrained(CHECKPOINT)
codec = HiggsAudioV2TokenizerModel.from_pretrained(CODEC).to(DEVICE).eval()
SAMPLE_RATE = codec.config.sample_rate  # 24 kHz

## Model calls

- `prefill` runs the prefix once and returns its keys and values.
- `log_probs` runs one pass over the target frames against that cache and returns the log-probability of every code in every slot. A target frame's input is the sum of its 8 codebook embeddings; a masked slot has an embedding of its own. `bias` is added to the target's attention scores towards the reference frames, in every layer and head.

In [ ]:
MASK = 1024  # the id of a masked slot, in every codebook


def embed_audio(codes: torch.Tensor) -> torch.Tensor:
    """[8, frames] codes -> [1, frames, hidden]: each frame is the sum of its 8 codebook embeddings."""
    offsets = 1025 * torch.arange(8, device=DEVICE)[:, None]  # codebook c owns embedding rows c * 1025 + code
    return model.audio_embeddings(codes + offsets).sum(0, keepdim=True)


def prefill(text: str, language: str, reference_text: str, reference: torch.Tensor) -> DynamicCache:
    """Runs the prefix once; returns its keys and values."""
    prompt = (
        f"<|denoise|><|lang_start|>{language}<|lang_end|><|instruct_start|>None<|instruct_end|>"
        f"<|text_start|>{reference_text} {text}<|text_end|>"
    )
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    prefix = torch.cat([model.llm.embed_tokens(ids), embed_audio(reference)], dim=1)
    length = prefix.size(1)
    cache = DynamicCache()
    # The mask has to be explicit: without one, the backbone falls back to causal attention.
    model.llm(inputs_embeds=prefix, attention_mask=torch.zeros(1, 1, length, length, device=DEVICE), past_key_values=cache)
    return cache


def log_probs(cache: DynamicCache, tokens: torch.Tensor, reference_frames: int, bias: float = 0.0) -> torch.Tensor:
    """One pass over the target: [8, frames] tokens -> [8, frames, 1024] log-probabilities."""
    prefix, frames = cache.get_seq_length(), tokens.size(1)
    mask = torch.zeros(1, 1, frames, prefix + frames, device=DEVICE)  # additive: 0 leaves a score as it is
    mask[..., prefix - reference_frames : prefix] = bias
    hidden = model.llm(
        inputs_embeds=embed_audio(tokens),
        attention_mask=mask,
        position_ids=torch.arange(prefix, prefix + frames, device=DEVICE)[None],
        past_key_values=cache,
    ).last_hidden_state
    cache.crop(prefix)  # the pass appended the target's keys and values; keep the prefix's only
    logits = model.audio_heads(hidden)[0].view(frames, 8, -1).transpose(0, 1)
    return logits.log_softmax(-1)[..., :MASK]  # a masked slot is never a prediction

## Sampler: sNcb2 + 2

In [ ]:
T_SHIFT = 0.1               # warps the commit schedule: few commits early, most in the last passes
CODEBOOK_PENALTY = 5.0      # per codebook index, so codebook 0 tends to be committed before 1
POSITION_TEMPERATURE = 5.0  # scales confidence and penalty against the Gumbel noise
REMASK_RATIO = 0.0          # earlier commits re-masked per new commit, on every main pass but the last; RL models run without
POLISH_BIAS = 1.0           # attention bias towards the reference frames, in the polish passes only
POLISH_GUIDANCE = 0.5       # codebooks 4-7: weight of (second polish pass - first polish pass)


def generate(cache: DynamicCache, frames: int, reference_frames: int, steps: int) -> torch.Tensor:
    """Fills a fully masked [8, frames] target: `steps` main passes over codebooks 0-1, then 2 polish passes."""
    tokens = torch.full((8, frames), MASK, device=DEVICE)
    codebook = torch.arange(8, device=DEVICE)[:, None]
    t = torch.linspace(0, 1, steps + 1)
    t = (T_SHIFT * t / (1 - (1 - T_SHIFT) * t)).tolist()

    for step in range(steps):
        confidence, prediction = log_probs(cache, tokens, reference_frames).max(-1)
        masked = (tokens == MASK) & (codebook < 2)
        revealed = (tokens != MASK) & (codebook < 2)
        last = step == steps - 1
        # This pass's share of the 2 * frames slots; the last pass fills whatever remasking left open.
        k = int(masked.sum()) if last else math.ceil(2 * frames * (t[step + 1] - t[step]))
        gumbel = -torch.log(-torch.log(torch.rand_like(confidence) + 1e-10) + 1e-10)
        score = (confidence - CODEBOOK_PENALTY * codebook) / POSITION_TEMPERATURE + gumbel
        chosen = score.masked_fill(~masked, -math.inf).flatten().topk(k).indices
        tokens.view(-1)[chosen] = prediction.view(-1)[chosen]
        if not last:  # remasking
            count = min(round(REMASK_RATIO * k), int(revealed.sum()))
            withdrawn = confidence.masked_fill(~revealed, math.inf).flatten().topk(count, largest=False).indices
            tokens.view(-1)[withdrawn] = MASK

    # Polish: codebooks 2-3, then 4-7 guided against the first polish pass's view of them.
    first = log_probs(cache, tokens, reference_frames, POLISH_BIAS)
    tokens[2:4] = first[2:4].argmax(-1)
    second = log_probs(cache, tokens, reference_frames, POLISH_BIAS)
    tokens[4:] = (second[4:] + POLISH_GUIDANCE * (second[4:] - first[4:])).argmax(-1)
    return tokens

## Target length and trimming

The target length is a rule-based estimate (OmniVoice's): the text's phonetic weight, spoken at the reference's rate. After decoding, the last 200 ms are cut, since the final frames can hold a short burst of noise, and leading and trailing silence is trimmed.

In [ ]:
def estimate_frames(text: str, reference_text: str, reference_frames: int) -> int:
    """The target's length: the text's spoken weight at the reference's rate (OmniVoice's rule, for alphabetic scripts)."""

    def weight(string: str) -> float:
        kinds = (unicodedata.category(char)[0] for char in string)
        return sum({"Z": 0.2, "M": 0.0, "P": 0.5, "S": 0.5, "N": 3.5}.get(kind, 1.0) for kind in kinds)

    frames = weight(text) / (weight(reference_text) / reference_frames)
    if frames < 50:  # short texts get relatively more time
        frames = 50 * (frames / 50) ** (1 / 3)
    return max(1, int(frames))


def trim_silence(audio: np.ndarray, sample_rate: int, keep: float = 0.1) -> np.ndarray:
    """Cuts leading and trailing silence (10 ms windows under -50 dBFS), keeping `keep` seconds of it on each side."""
    window = sample_rate // 100
    count = len(audio) // window
    level = 20 * np.log10(np.sqrt((audio[: count * window].reshape(count, window) ** 2).mean(1)) + 1e-10)
    loud = np.flatnonzero(level >= -50)
    if len(loud) == 0:
        return audio
    sounds = np.split(loud, np.flatnonzero(np.diff(loud) > 20) + 1)  # separated by over 200 ms of silence
    # A sound of at most 120 ms, alone at either edge, is a codec burst rather than speech.
    if len(sounds) > 1 and len(sounds[0]) <= 12:
        sounds = sounds[1:]
    if len(sounds) > 1 and len(sounds[-1]) <= 12:
        sounds = sounds[:-1]
    margin = round(keep * 100)
    return audio[max(sounds[0][0] - margin, 0) * window : (sounds[-1][-1] + 1 + margin) * window]

## Reference

The recording is mixed down to mono, resampled to 24 kHz and zero-padded to whole codec frames, as the training codes were. For about half of all other lengths the codec's encoder pads its acoustic input by half a frame, and the codes come out misaligned.

In [ ]:
wav, rate = sf.read(REFERENCE, dtype="float32", always_2d=True)  # [samples, channels]
wav = torchaudio.functional.resample(torch.from_numpy(wav.mean(1))[None], rate, SAMPLE_RATE)  # [1, samples]
wav = torch.nn.functional.pad(wav, (0, -wav.size(1) % codec.config.hop_length))
reference = codec.encode(wav[None].to(DEVICE)).audio_codes[0]  # [8, frames]

## Generate

In [ ]:
frames = estimate_frames(TEXT, REFERENCE_TEXT, reference.size(1))

cache = prefill(TEXT, LANGUAGE, REFERENCE_TEXT, reference)
torch.manual_seed(SEED)
tokens = generate(cache, frames, reference.size(1), STEPS)

audio = codec.decode(tokens[None]).audio_values[0, 0].cpu().numpy()
audio = trim_silence(audio[: -int(0.2 * SAMPLE_RATE)], SAMPLE_RATE)
sf.write(OUTPUT, audio, SAMPLE_RATE)
Audio(audio, rate=SAMPLE_RATE)